In [26]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn import linear_model
from tqdm import tqdm
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split


In [27]:
path = 'data/'

data_X = pd.read_csv(path + 'X_train.csv',index_col='ROW_ID')
data_y = pd.read_csv(path + 'y_train.csv',index_col='ROW_ID')
X_submission = pd.read_csv(path + 'X_test.csv',index_col='ROW_ID')
sample_submission = pd.read_csv(path + 'sample_submission.csv',index_col='ROW_ID')

date_train, date_test = train_test_split(data_X['TS'].unique(), test_size=0.05, random_state=42)

X_train = data_X[data_X['TS'].isin(date_train)].copy()
y_train = data_y.loc[X_train.index].copy()

X_test = data_X[data_X['TS'].isin(date_test)].copy()
y_test = data_y.loc[X_test.index].copy()

RET_features = [f'RET_{i}' for i in range(1,21)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,21)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']

scaler = StandardScaler()

# Dictionnaire des groupes de features
feature_groups = {
    'RET_features': RET_features,
    'SIGNED_VOLUME_features': SIGNED_VOLUME_features,
    'TURNOVER': ['AVG_DAILY_TURNOVER']
}

# Boucle pour scaler chaque groupe
for group_name, cols in feature_groups.items():
    # Fit sur le train, transform sur train, test et submission
    X_train[cols] = scaler.fit_transform(X_train[cols])
    X_test[cols] = scaler.transform(X_test[cols])
    X_submission[cols] = scaler.transform(X_submission[cols])



FEATURES

In [28]:
EPS = 1e-12

def ema(arr, L):
    alpha = 2/(L+1)
    w = (1-alpha) ** np.arange(L)  # 0..L-1
    w = w / w.sum()
    return (arr[:, :L] * w).sum(axis=1)

def last_streak_len(sig_row, positive=True):
    # part de RET_1 vers RET_20
    target = 1 if positive else -1
    cnt = 0
    for v in sig_row[:20]:  # [:20] explicite
        if v == target:
            cnt += 1
        else:
            break
    return cnt


def maxdd_and_recovery(row):
    c = np.cumsum(row)                   # equity curve 20j
    peak = np.maximum.accumulate(c)
    dd = (c - peak)
    maxdd = dd.min()                     # drawdown (négatif)
    # recovery: jours depuis le dernier pic
    last_peak_idx = np.where(c == peak)[0][-1]
    recovery = 20 - 1 - last_peak_idx
    return maxdd, recovery

def add_features(df):
    for i in range(1, 21):
        df[f'FAR_{i}'] = df[f'RET_{i}']*(df[f'SIGNED_VOLUME_{i}']) / df[SIGNED_VOLUME_features].abs().mean(axis=1)
        df[f'FAR_{i}_adjusted'] = df[f'FAR_{i}'] * (df['AVG_DAILY_TURNOVER'])

    for i in [3,5,10,15,20]:
        df[ f'AVERAGE_PERF_{i}'] = df[RET_features[:i]].mean(1)
        df[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = df.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')

    r = df[RET_features].to_numpy()
    df["ema3"]  = ema(r, 3)
    df["ema5"]  = ema(r, 5)
    df["ema10"] = ema(r,10)
    m20 = r.mean(axis=1)
    s20 = r.std(axis=1, ddof=0)

    df["z20"] = m20 / (s20 + 1e-12)
    sign = np.sign(r)  

    df["streak_pos"] = [last_streak_len(s, True) for s in sign]
    df["streak_neg"] = [last_streak_len(s, False) for s in sign]

    
    p = (r > 0).mean(axis=1)
    p = np.clip(p, 1e-9, 1 - 1e-9)
    df["sign_entropy20"] = -(p*np.log(p) + (1-p)*np.log(1-p))

    
    r10 = r[:, :10]
    m10 = r10.mean(axis=1)
    s10 = r10.std(axis=1, ddof=0)
    neg10 = np.minimum(r10, 0)

    df["sharpe10"]   = m10 / (s10 + 1e-12)
    downside10 = np.sqrt((neg10**2).mean(axis=1))
    df["sortino10"]  = m10 / (downside10 + 1e-12)
    df["tstat_mean10"] = m10 / (s10/np.sqrt(10) + 1e-12)
    
    df["vol10"]         = s10
    df["downside_dev10"]= downside10

    md_rec = np.apply_along_axis(maxdd_and_recovery, 1, r)
    df["maxdd20"]   = md_rec[:,0]
    df["recovery20"]= md_rec[:,1]


    v = np.nan_to_num(df[SIGNED_VOLUME_features].to_numpy(), nan=0.0)
    r = np.nan_to_num(df[RET_features].to_numpy(), nan=0.0)

    df["sv_mean20"] = v.mean(axis=1)
    df["sv_std20"]  = v.std(axis=1, ddof=0)

    v_mean = v.mean(axis=1, keepdims=True)
    vx = v - v_mean
    num = (vx[:,1:] * vx[:,:-1]).sum(axis=1)
    den = np.sqrt((vx[:,1:]**2).sum(axis=1) * (vx[:,:-1]**2).sum(axis=1))
    df["sv_autocorr1"] = num / (den + EPS)

    rx = r - r.mean(axis=1, keepdims=True)
    num = (rx * vx).sum(axis=1)
    den = np.sqrt((rx**2).sum(axis=1) * (vx**2).sum(axis=1))
    df["corr_ret_sv20"] = num / (den + EPS)


    return df



In [29]:

X_train = add_features(X_train)
X_test = add_features(X_test)
X_submission = add_features(X_submission)


features = X_train.select_dtypes(include=[np.floating]).columns.tolist()


C:\Users\maloc\AppData\Local\Temp\ipykernel_11556\2373366857.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["sortino10"]  = m10 / (downside10 + 1e-12)
C:\Users\maloc\AppData\Local\Temp\ipykernel_11556\2373366857.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["tstat_mean10"] = m10 / (s10/np.sqrt(10) + 1e-12)
C:\Users\maloc\AppData\Local\Temp\ipykernel_11556\2373366857.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

In [30]:
def add_top_features(df, names_top_features):
    new_features = {}

    for f in names_top_features:
        parts = f.replace('^2', '').split(' ')
        if len(parts) == 1 and '^2' in f:
            col_name = f"{parts[0]}^2"
            new_features[col_name] = df[parts[0]] ** 2
        elif len(parts) == 2:
            col_name = f"{parts[0]} {parts[1]}"
            new_features[col_name] = df[parts[0]] * df[parts[1]]
        # print(f"Computed feature: {col_name} base columns: {f}")

    # Convert to DataFrame and concatenate all at once
    df_new = pd.concat([df, pd.DataFrame(new_features, index=df.index)], axis=1)
    return df_new

Features selection : 

In [31]:
ALPHAS = np.logspace(2, 5, 11)
def opti_alpha(features, degree=1):
    best_alpha = None
    best_r2 = -np.inf
    best_feature_names = None
    best_coefs = None
    
    pipeline = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error", fit_intercept=True)
    )

    # Fit model
    pipeline.fit(X_train.loc[:,features], y_train)

    # Evaluate
    r2 = pipeline.score(X_test.loc[:,features], y_test)

    
    best_alpha = pipeline.named_steps['ridgecv'].alpha_
    best_feature_names = pipeline.named_steps['polynomialfeatures'].get_feature_names_out()
    best_coefs = pipeline.named_steps['ridgecv'].coef_

    print(f"Alpha: {best_alpha}, R² score: {best_r2:.3f}")
    

    filtered = [(name, coef) for name, coef in zip(best_feature_names, best_coefs)]
    filtered = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)

    return best_alpha, filtered

best_alpha, filtered = opti_alpha( features, 1)
top_n = 50
final_features = [name for name, coef in filtered[:top_n]]


X_train = add_top_features(X_train, final_features)
X_test = add_top_features(X_test, final_features)
X_submission = add_top_features(X_submission, final_features)



c:\Users\maloc\.conda\envs\qrt_env\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=7.31814e-25): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\maloc\.conda\envs\qrt_env\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=6.39331e-25): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\maloc\.conda\envs\qrt_env\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=6.38146e-25): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\maloc\.conda\envs\qrt_env\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.17577e-24): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
c:\Users\maloc\.conda\envs\qrt_env\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=7.23633e-25): result may not be accurate

Alpha: 100000.0, R² score: -inf


c:\Users\maloc\.conda\envs\qrt_env\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.80441e-22): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


In [32]:
best_alpha, filtered = opti_alpha(final_features, 2)
top_n = 30
final_features = [name for name, coef in filtered[:top_n]]


X_train = add_top_features(X_train, final_features)
X_test = add_top_features(X_test, final_features)
X_submission = add_top_features(X_submission, final_features)

Alpha: 100000.0, R² score: -inf


Sort absolute coef

In [33]:
model = make_pipeline(
    Ridge(alpha=best_alpha, max_iter=10000)
)

# Fit model
model.fit(X_train.loc[:,final_features], y_train)


,steps,"[('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,alpha,np.float64(100000.0)
,fit_intercept,True
,copy_X,True
,max_iter,10000
,tol,0.0001
,solver,'auto'
,positive,False


In [34]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model,
    X_test.loc[:,final_features],
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='r2'
)
# Créer un DataFrame des importances
perm_importance_df = pd.DataFrame({
    'feature': final_features,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
})

# Trier par importance moyenne
perm_importance_df = perm_importance_df.sort_values(by='importance_mean', ascending=False)

final_features = perm_importance_df[perm_importance_df['importance_mean'] > 0]['feature'].tolist()


Perceptron

In [35]:
X_train_perceptron = X_train.loc[:, final_features]
y_train_bin = (y_train > 0).astype(int)
X_test_perceptron = X_test.loc[:, final_features]
y_test_bin = (y_test > 0).astype(int)

shape = X_train_perceptron.shape

# Créer le modèle avec Dropout
model = Sequential([
    Dense(16, activation='relu', input_shape=(shape[1],)),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Compiler
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping pour éviter l'overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entraînement
history = model.fit(
    X_train_perceptron, y_train_bin,
    validation_data=(X_test_perceptron, y_test_bin),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/100


c:\Users\maloc\.conda\envs\qrt_env\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5351/5351 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.5063 - loss: 0.7010 - val_accuracy: 0.5348 - val_loss: 0.6928
Epoch 2/100
5351/5351 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.5138 - loss: 0.6929 - val_accuracy: 0.5405 - val_loss: 0.6910
Epoch 3/100
5351/5351 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.5172 - loss: 0.6924 - val_accuracy: 0.5328 - val_loss: 0.6909
Epoch 4/100
5351/5351 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.5175 - loss: 0.6922 - val_accuracy: 0.5292 - val_loss: 0.6907
Epoch 5/100
5351/5351 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.5181 - loss: 0.6923 - val_accuracy: 0.5349 - val_loss: 0.6898
Epoch 6/100
5351/5351 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.5197 - loss: 0.6922 - val_accuracy: 0.5303 - val_loss: 0.6898
Epoch 7/100
5351/5351 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.5208 - loss: 0.6921 - val_accuracy: 0.5359 - val_loss: 0.6901
Epoch 8/100
5351/5351 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.5204 - loss: 0.6919 - val_

In [36]:
# 5️⃣ Évaluation


y_pred = (model.predict(X_test_perceptron) > 0.5).astype(int)
acc = accuracy_score(y_test_bin, y_pred)
print(f"Accuracy: {acc:.3f}")

283/283 ━━━━━━━━━━━━━━━━━━━━ 0s 774us/step
Accuracy: 0.535


PREDICTION

In [37]:
X_submission_perceptron = X_submission.loc[:, final_features]
y_submission = model.predict(X_submission_perceptron)
y_submission = pd.DataFrame(y_submission, index=sample_submission.index,columns=['target'])
(y_submission>0).astype(int).to_csv('predict/y_submission.csv')

242/242 ━━━━━━━━━━━━━━━━━━━━ 0s 662us/step
